In [1]:
# Import libraries for loading
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np

# ---------------------------
# Load the saved model and scaler
# ---------------------------
# Load the trained Keras model
loaded_model = load_model("my_model.h5")
print("Model loaded successfully.")

# Load the scaler from pickle file
with open("scaler.pkl", "rb") as f:
    loaded_scaler = pickle.load(f)
print("Scaler loaded successfully.")

# ---------------------------
# Example: Testing the Model on New Data
# ---------------------------
# Assume new_sample is a new 1D numpy array of reflectance values with the same number of features as the model input.
# Here we create a fake sample for demonstration (replace with your actual data).
new_sample = np.array([0.2, 0.5, 0.8])  # Example - ensure its length equals number of features
if new_sample.shape[0] != loaded_model.input_shape[1]:
    raise ValueError(f"Expected sample with {loaded_model.input_shape[1]} features, got {new_sample.shape[0]}.")

# Scale the sample using the loaded scaler
new_sample_scaled = loaded_scaler.transform(new_sample.reshape(1, -1))

# Predict water quality parameters using the loaded model
prediction = loaded_model.predict(new_sample_scaled)
print("Predicted water quality parameters for new sample:", prediction)

# You can further integrate this testing part with your process for reading .nc files if needed.


TypeError: Could not locate function 'mse'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'module': 'keras.metrics', 'class_name': 'function', 'config': 'mse', 'registered_name': 'mse'}

In [2]:
# Import necessary libraries for loading and testing
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np

# ---------------------------
# Step 1: Load the Saved Model and Scaler
# ---------------------------
# When loading, we provide custom_objects to map the string 'mse' to the MeanSquaredError metric.
from tensorflow.keras.metrics import MeanSquaredError
loaded_model = load_model("my_model.h5", custom_objects={"mse": MeanSquaredError()})
print("Model loaded successfully.")

with open("scaler.pkl", "rb") as f:
    loaded_scaler = pickle.load(f)
print("Scaler loaded successfully.")

# ---------------------------
# Step 2: Test the Model on a New Sample
# ---------------------------
# For demonstration, create a new sample. In practice, replace this with your actual new data.
# Make sure the new sample has the same number of features as used during training.
new_sample = np.array([0.25, 0.55, 0.85])  # Replace with real data; length should equal X_train.shape[1]

if new_sample.shape[0] != loaded_model.input_shape[1]:
    raise ValueError(f"Expected sample with {loaded_model.input_shape[1]} features, got {new_sample.shape[0]}.")

# Scale the new sample (scaler expects 2D input)
new_sample_scaled = loaded_scaler.transform(new_sample.reshape(1, -1))

# Predict water quality parameters using the loaded model
prediction = loaded_model.predict(new_sample_scaled)
print("Predicted water quality parameters for new sample:", prediction)


Model loaded successfully.
Scaler loaded successfully.


ValueError: Expected sample with 551 features, got 3.

In [ ]:
# Import necessary libraries for loading and testing
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np

# ---------------------------
# Step 1: Load the Saved Model and Scaler
# ---------------------------
# When loading the model, we supply custom_objects to map the string 'mse' 
# (used during compilation) to the MeanSquaredError metric.
from tensorflow.keras.metrics import MeanSquaredError
loaded_model = load_model("my_model.h5", custom_objects={"mse": MeanSquaredError()})
print("Model loaded successfully.")

with open("scaler.pkl", "rb") as f:
    loaded_scaler = pickle.load(f)
print("Scaler loaded successfully.")

# ---------------------------
# Step 2: Test the Model on a New Sample
# ---------------------------
# For demonstration, create a new sample. In practice, replace this with your actual new data.
# (Make sure the new sample has the same number of features as your model’s input.)
new_sample = np.array([0.25, 0.55, 0.85])  # For example; its length should equal X_train.shape[1]

if new_sample.shape[0] != loaded_model.input_shape[1]:
    raise ValueError(f"Expected sample with {loaded_model.input_shape[1]} features, got {new_sample.shape[0]}.")

# Scale the new sample (the scaler expects a 2D input)
new_sample_scaled = loaded_scaler.transform(new_sample.reshape(1, -1))

# Predict water quality parameters using the loaded model
prediction = loaded_model.predict(new_sample_scaled)
print("Predicted water quality parameters for new sample:", prediction)


Model loaded successfully.
Scaler loaded successfully.


ValueError: Expected sample with 551 features, got 3.

In [5]:
# Import necessary libraries for testing
import os
import xarray as xr
import numpy as np

# ----------------------------------------------------------------
# Configuration: Set the directory path for Sentinel-3 OLCI .nc files
# ----------------------------------------------------------------
olci_nc_directory = "C:/Users/BraedynM/Documents/Science Fair 2025/OLCI_dataset"  # Update this to your actual directory

# ----------------------------------------------------------------
# Retrieve a filtered list of NetCDF files in the directory that start with "Oa"
# This ensures ancillary files (e.g., geo_coordinates.nc) are skipped.
# ----------------------------------------------------------------
try:
    nc_files = [f for f in os.listdir(olci_nc_directory) if f.startswith("Oa") and f.endswith('.nc')]
except FileNotFoundError:
    raise FileNotFoundError(f"Directory '{olci_nc_directory}' not found. Please update the path.")

# ----------------------------------------------------------------
# Determine the expected number of features (spectral bands)
# based on the trained model's CNN input shape.
# For example, if the model was trained with inputs of shape (num_features, 1):
# ----------------------------------------------------------------
expected_n_features = loaded_model.input_shape[1]  # e.g., if model.input_shape is (None, 21, 1), expected_n_features == 21

print("\nTesting OLCI .nc files with the trained CNN model:")

for nc_file in nc_files:
    file_path = os.path.join(olci_nc_directory, nc_file)
    try:
        # Open the NetCDF file using xarray
        ds = xr.open_dataset(file_path)
    except Exception as e:
        print(f"Error opening '{nc_file}': {e}")
        continue

    # Instead of looking for a 'reflectance' variable, look for a variable that contains "radiance"
    radiance_vars = [var for var in ds.variables if "radiance" in var.lower()]
    if not radiance_vars:
        print(f"File '{nc_file}' does not contain a radiance variable. Skipping.")
        continue

    # For this example, we choose the first radiance variable found.
    data_var = radiance_vars[0]
    data_vals = ds[data_var].values

    # If the variable name includes "radiance", we assume a simple conversion.
    # (Replace the conversion_factor below with a proper TOA reflectance conversion as needed.)
    conversion_factor = 1.0  # Placeholder: set to the appropriate factor for your data
    sample_data = data_vals * conversion_factor

    # If the data is multi-dimensional (e.g., height x width x bands), average over spatial dimensions.
    if sample_data.ndim == 3:
        # Assuming the last dimension corresponds to spectral bands.
        sample = np.mean(sample_data, axis=(0, 1))
    elif sample_data.ndim == 2:
        # If already 2D (e.g., pixels x bands), average over the pixel dimension.
        sample = np.mean(sample_data, axis=0)
    elif sample_data.ndim == 1:
        sample = sample_data
    else:
        print(f"File '{nc_file}' has unsupported dimensions (ndim = {sample_data.ndim}). Skipping.")
        continue

    # Verify that the resulting sample has the expected number of spectral bands/features.
    if sample.shape[0] != expected_n_features:
        print(f"File '{nc_file}' has {sample.shape[0]} features; expected {expected_n_features}. Skipping.")
        continue

    # Scale the sample using the same scaler fitted during training.
    # The scaler expects a 2D input so reshape to (1, -1)
    sample_scaled = loaded_scaler.transform(sample.reshape(1, -1))
    
    # Reshape the sample for the CNN model (adding the channel dimension):
    # Final shape will be (1, expected_n_features, 1)
    sample_ready = sample_scaled.reshape(1, expected_n_features, 1)

    # Predict water quality parameters using the trained CNN model
    prediction = loaded_model.predict(sample_ready)
    print(f"File: {nc_file} --> Predicted water quality parameters: {prediction}")



Testing OLCI .nc files with the trained CNN model:
File 'Oa01_radiance.nc' has 4865 features; expected 551. Skipping.
File 'Oa01_radiance_unc.nc' has 4865 features; expected 551. Skipping.
File 'Oa02_radiance.nc' has 4865 features; expected 551. Skipping.
File 'Oa02_radiance_unc.nc' has 4865 features; expected 551. Skipping.
File 'Oa03_radiance.nc' has 4865 features; expected 551. Skipping.
File 'Oa03_radiance_unc.nc' has 4865 features; expected 551. Skipping.
File 'Oa04_radiance.nc' has 4865 features; expected 551. Skipping.
File 'Oa04_radiance_unc.nc' has 4865 features; expected 551. Skipping.
File 'Oa05_radiance.nc' has 4865 features; expected 551. Skipping.
File 'Oa05_radiance_unc.nc' has 4865 features; expected 551. Skipping.
File 'Oa06_radiance.nc' has 4865 features; expected 551. Skipping.
File 'Oa06_radiance_unc.nc' has 4865 features; expected 551. Skipping.
File 'Oa07_radiance.nc' has 4865 features; expected 551. Skipping.
File 'Oa07_radiance_unc.nc' has 4865 features; expect

In [6]:
# Import necessary libraries for testing
import os
import xarray as xr
import numpy as np

# ----------------------------------------------------------------
# Configuration: Set the directory path for Sentinel-3 OLCI .nc files
# ----------------------------------------------------------------
olci_nc_directory = "C:/Users/BraedynM/Documents/Science Fair 2025/OLCI_dataset"  # provided path

# ----------------------------------------------------------------
# Retrieve a filtered (and sorted) list of NetCDF files that start with "Oa"
# This way ancillary files (e.g., geo_coordinates.nc) are skipped.
# ----------------------------------------------------------------
try:
    # Here we assume that files starting with "Oa" represent one spectral band each.
    nc_files = [f for f in os.listdir(olci_nc_directory) if f.startswith("Oa") and f.endswith('.nc')]
    # Sort files alphabetically (or by band number if the naming convention supports it)
    nc_files = sorted(nc_files)
except FileNotFoundError:
    raise FileNotFoundError(f"Directory '{olci_nc_directory}' not found. Please update the path.")

# ----------------------------------------------------------------
# Determine the expected number of spectral features from the loaded model
# For example, if the model was trained with inputs of shape (num_features, 1),
# then expected_n_features should be that number.
# ----------------------------------------------------------------
expected_n_features = loaded_model.input_shape[1]  
print(f"Expected spectral bands (features): {expected_n_features}")

# ----------------------------------------------------------------
# Process each file to extract a scalar value per spectral band
# ----------------------------------------------------------------
spectral_values = []  # to accumulate one scalar per valid file

for nc_file in nc_files:
    file_path = os.path.join(olci_nc_directory, nc_file)
    try:
        ds = xr.open_dataset(file_path)
    except Exception as e:
        print(f"Error opening '{nc_file}': {e}")
        continue

    # Instead of looking for a variable 'reflectance', we first look for a variable that contains "radiance"
    radiance_vars = [var for var in ds.variables if "radiance" in var.lower()]
    if not radiance_vars:
        print(f"File '{nc_file}' does not contain a radiance variable. Skipping.")
        continue

    # Assume the first radiance variable is the one we want
    data_var = radiance_vars[0]
    data_vals = ds[data_var].values

    # If the data is multi-dimensional (e.g., height x width), take the spatial average.
    # For a 2D array, average over axis=0 and/or axis=1 so that the output is a scalar.
    if data_vals.ndim >= 2:
        value = np.mean(data_vals)  # average over all pixels
    elif data_vals.ndim == 1:
        value = np.mean(data_vals)  # even for 1D, get a representative number
    else:
        print(f"File '{nc_file}' has unsupported dimensions (ndim = {data_vals.ndim}). Skipping.")
        continue

    spectral_values.append(value)

# Convert the list to a numpy array.
spectral_vector = np.array(spectral_values)  # shape should be (num_valid_files,)

print(f"Extracted spectral vector has shape: {spectral_vector.shape}")

# Check if we got the expected number of features.
if spectral_vector.shape[0] != expected_n_features:
    print(f"Error: Combined spectral vector has {spectral_vector.shape[0]} features; expected {expected_n_features}.")
else:
    # Scale the spectral vector using the loaded scaler.
    # The scaler expects 2D input, so reshape accordingly.
    spectral_vector_scaled = loaded_scaler.transform(spectral_vector.reshape(1, -1))
    
    # Reshape for the CNN model. 
    # Final shape should be (1, expected_n_features, 1).
    sample_ready = spectral_vector_scaled.reshape(1, expected_n_features, 1)
    
    # Predict water quality parameters using the loaded model.
    prediction = loaded_model.predict(sample_ready)
    print("Predicted water quality parameters for the spectral vector:")
    print(prediction)


Expected spectral bands (features): 551
Extracted spectral vector has shape: (42,)
Error: Combined spectral vector has 42 features; expected 551.


In [7]:
# Import necessary libraries for testing
import os
import xarray as xr
import numpy as np

# ----------------------------------------------------------------
# Configuration: Set the directory path for Sentinel-3 OLCI .nc files
# ----------------------------------------------------------------
olci_nc_directory = "C:/Users/BraedynM/Documents/Science Fair 2025/OLCI_dataset"

# ----------------------------------------------------------------
# Retrieve all NetCDF files in the directory (remove the "Oa" filter)
# ----------------------------------------------------------------
try:
    nc_files = [f for f in os.listdir(olci_nc_directory) if f.endswith('.nc')]
    nc_files = sorted(nc_files)  # sort alphabetically (or by a known band order)
    print(f"Found {len(nc_files)} .nc files.")
except FileNotFoundError:
    raise FileNotFoundError(f"Directory '{olci_nc_directory}' not found. Please update the path.")

# ----------------------------------------------------------------
# Determine the expected number of features based on the loaded model
# ----------------------------------------------------------------
expected_n_features = loaded_model.input_shape[1]  # for example, 551
print(f"Expected spectral bands (features): {expected_n_features}")

# ----------------------------------------------------------------
# Process each file to extract a scalar value per spectral band
# ----------------------------------------------------------------
spectral_values = []  # accumulator for one scalar per file

for nc_file in nc_files:
    file_path = os.path.join(olci_nc_directory, nc_file)
    try:
        # Open the NetCDF file using xarray
        ds = xr.open_dataset(file_path)
    except Exception as e:
        print(f"Error opening '{nc_file}': {e}")
        continue

    # Look for a variable containing "radiance" (case-insensitive)
    radiance_vars = [var for var in ds.variables if "radiance" in var.lower()]
    if not radiance_vars:
        print(f"File '{nc_file}' does not contain a radiance variable. Skipping.")
        continue

    # Use the first radiance variable found
    data_var = radiance_vars[0]
    data_vals = ds[data_var].values

    # Apply a conversion factor if needed (set here to 1.0 as a placeholder)
    conversion_factor = 1.0  # Replace with a proper conversion if required
    sample_data = data_vals * conversion_factor

    # Average over spatial dimensions
    if sample_data.ndim >= 2:
        value = np.mean(sample_data)  # average over all pixels/dimensions
    elif sample_data.ndim == 1:
        value = np.mean(sample_data)
    else:
        print(f"File '{nc_file}' has unsupported dimensions (ndim = {sample_data.ndim}). Skipping.")
        continue

    spectral_values.append(value)

# Build the complete spectral vector from the accumulated values.
spectral_vector = np.array(spectral_values)  # shape: (number_of_valid_files,)

print(f"Extracted spectral vector has shape: {spectral_vector.shape}")

# If the combined spectral vector does not have the expected number of features...
if spectral_vector.shape[0] != expected_n_features:
    print(f"Error: Combined spectral vector has {spectral_vector.shape[0]} features; expected {expected_n_features}.")
    # Option 1: If you expected 551 files, then the file filtering needs to be adjusted.
    # Option 2: Otherwise, you may need to retrain or modify the model for the available bands.
else:
    # Scale the spectral vector using the loaded scaler (scaler expects a 2D array)
    spectral_vector_scaled = loaded_scaler.transform(spectral_vector.reshape(1, -1))
    
    # Reshape for the CNN model: final shape should be (1, expected_n_features, 1)
    sample_ready = spectral_vector_scaled.reshape(1, expected_n_features, 1)
    
    # Predict water quality parameters using the loaded model
    prediction = loaded_model.predict(sample_ready)
    print(f"Predicted water quality parameters for the spectral vector from the OLCI dataset:")
    print(prediction)


Found 50 .nc files.
Expected spectral bands (features): 551
File 'geo_coordinates.nc' does not contain a radiance variable. Skipping.
File 'instrument_data.nc' does not contain a radiance variable. Skipping.
File 'qualityFlags.nc' does not contain a radiance variable. Skipping.
File 'tie_geo_coordinates.nc' does not contain a radiance variable. Skipping.
File 'tie_geometries.nc' does not contain a radiance variable. Skipping.
File 'tie_meteo.nc' does not contain a radiance variable. Skipping.


c:\Users\BraedynM\AppData\Local\Programs\Python\Python312\Lib\site-packages\xarray\namedarray\core.py:264: UserWarning: Duplicate dimension names present: dimensions {'bands'} appear more than once in dims=('bands', 'bands'). We do not yet support duplicate dimension names, but we do allow initial construction of the object. We recommend you rename the dims immediately to become distinct, as most xarray functionality is likely to fail silently if you do not. To rename the dimensions you will need to set the ``.dims`` attribute of each variable, ``e.g. var.dims=('x0', 'x1')``.
  self._dims = self._parse_dimensions(dims)
c:\Users\BraedynM\AppData\Local\Programs\Python\Python312\Lib\site-packages\xarray\namedarray\core.py:264: UserWarning: Duplicate dimension names present: dimensions {'bands'} appear more than once in dims=('bands', 'bands'). We do not yet support duplicate dimension names, but we do allow initial construction of the object. We recommend you rename the dims immediately t

File 'time_coordinates.nc' does not contain a radiance variable. Skipping.
Extracted spectral vector has shape: (43,)
Error: Combined spectral vector has 43 features; expected 551.
